This is the initial script to create the hamiltonian dataset using the molecular geometries provided by the W4 dataset, using this library to iterate over the data.

This first function is just the script imports, and a helper function to return a struct from a base64 encoded numpy tensor.

In [1]:
import base64, json, numpy as np
from pyscf import gto, scf, mcscf, ao2mo, cc
from pyscf.data.elements import chemcore
from w4benchmark import W4Decorators, Molecule, W4

hamiltonian_dict = {}

def serialize_tensor(tensor: np.ndarray):
    if isinstance(tensor, tuple): return [serialize_tensor(i) for i in tensor]
    return {
        "dtype": str(tensor.dtype),
        "shape": tensor.shape,
        "data": base64.b64encode(tensor.tobytes()).decode("utf-8")
    }

This next function returns a map of values parsed and computed from the geometry data.

In [2]:
def gen_hamiltonians(molecule: Molecule):
    # Create pyScf mol from w4benchmark molecule
    scfMol = gto.Mole()
    scfMol.atom = molecule.geom
    scfMol.basis = W4.parameters.basis
    scfMol.symmetry = False
    scfMol.charge = molecule.charge
    scfMol.spin = int(molecule.spin - 1)
    scfMol.verbose = 5
    scfMol.build()

    # Run Hartree Fock on all-electron space
    if 0 == scfMol.spin: mf = scf.RHF(scfMol)
    else: mf = scf.ROHF(scfMol)

    mf.kernel()
    if not mf.converged:
        print("converge failed: %s", molecule.species)

    # Active space computations

    # Freeze core
    nao = scfMol.nao_nr()
    ncore = chemcore(scfMol)
    nelec_as = tuple(nelec - ncore for nelec in scfMol.nelec)  # KEY2
    ncas = nao - ncore  # KEY1
    active_orbs = [p for p in range(ncore, nao)]
    frozen_orbs = [i for i in range(nao) if i not in active_orbs]

    open_shell = not (nelec_as[0] == nelec_as[1])

    np_orbs = np.array(active_orbs)
    # Post-Hartree calculations
    ## Hartree-Fock SCF solutions: basis functions (eigenvectors of Fock operator)
    ##   defined as linear combinations of the basis set functions
    ## Reference state: Hartree-Fock ground state
    # Get molecular coefficients of basis functions from Hartree Fock results (Slater determinants)
    mo = mf.mo_coeff
    # Form subset of Slater determinants in active space by picking out the corresponding coefficients
    # Note: the Hartree-Fock solutions are automatically ordered/sorted according to eigenvalues by PySCF
    h1e_cas = ecore = h2e_cas = mf_cas = None
    if isinstance(mf, scf.hf.RHF) or isinstance(mf, scf.rohf.ROHF):
        mf_cas = mcscf.CASSCF(mf, ncas, nelec_as)
        h1e_cas, ecore = mf_cas.get_h1eff()
        eri = mf_cas.mol.intor('int2e')  # This is not working for mo_cas
        h2e = ao2mo.incore.full(eri, mo)
        h2e_cas = h2e[np.ix_(np_orbs, np_orbs, np_orbs, np_orbs)]
    elif isinstance(mf, scf.uhf.UHF):
        mf_cas = mcscf.UCASSCF(mf, ncas, nelec_as)
        h1e_cas, ecore = mf_cas.get_h1eff()
        h2e_cas = mf_cas.get_h2eff()

    # cc_as = cc.CCSD(mf).run() if len(frozen_orbs) == 0 else cc.CCSD(mf, frozen=frozen_orbs).run()

    return {
        "ecore": str(ecore),
        "ncas": mf_cas.ncas,
        "nelecas": mf_cas.nelecas,
        "h1e": serialize_tensor(h1e_cas),
        "h2e": serialize_tensor(h2e_cas),
        # "cct2": serialize_tensor(cc_as.t2)
    }

Then, the penultimate step is to iterate over all 152 molecular geometries provided by the W4 dataset and generate all corresponding information. This implementation leverages the '--process' flag that iterates and supplies the molecules for portable and fast iteration.

In [3]:
@W4Decorators.process(basis="sto6g")
def iter_gen(name: str, mol: Molecule):
    print(f"\n\Generating {name}:")
    hamiltonian_dict[name] = gen_hamiltonians(mol)

Finally, all that's needed is to output the information to file, which is done as static code executed after the decorated function is done iterating. One could also forego the CLI flag approach for explicitly dereferencing the W4 singleton and iterating over the data manually.

In [ ]:
if __name__ == '__main__':
    # to run without CLI arg "--process"
    W4.parameters.basis = "sto6g"
    W4.init()
    for species, mol in W4:
        iter_gen(species, mol)
    # otherwise, only the file IO is necessary
    with open("hamiltonian_dataset.json", "w") as f:
        json.dump(hamiltonian_dict, f, indent=4)
    print("Finished calculating hamiltonians.")


\Generating acetaldehyde:
System: uname_result(system='Linux', node='LukasStrix18', release='6.6.87.2-microsoft-standard-WSL2', version='#1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025', machine='x86_64')  Threads 32
Python 3.10.12 (main, Nov  4 2025, 08:48:33) [GCC 11.4.0]
numpy 2.2.4  scipy 1.15.2  h5py 3.13.0
Date: Fri Dec 12 21:17:30 2025
PySCF version 2.8.0
PySCF path  /home/lpetervary/.virtualenvs/w4benchmark/lib/python3.10/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 5
[INPUT] max_memory = 4000 
[INPUT] num. atoms = 7
[INPUT] num. electrons = 23
[INPUT] charge = 1
[INPUT] spin (= nelec alpha-beta = 2S) = -1
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 C      0.000000000000   0.000000000000   0.000000000000 AA    0.000000000000   0.000000000000   0.000000000000 Bohr   0.0
[INPUT]  2 O      0.0